SETUP

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

RAW = Path("../data/raw")

files = {
    "orders":       "olist_orders_dataset.csv",
    "order_items":  "olist_order_items_dataset.csv",
    "payments":     "olist_order_payments_dataset.csv",
    "reviews":      "olist_order_reviews_dataset.csv",
    "customers":    "olist_customers_dataset.csv",
    "sellers":      "olist_sellers_dataset.csv",
    "products":     "olist_products_dataset.csv",
    "geolocation":  "olist_geolocation_dataset.csv",
    "translation":  "product_category_name_translation.csv",
}

def profile(name, df):
    print(f"\n{'='*65}")
    print(f"  {name.upper()}")
    print(f"{'='*65}")
    print(f"  Shape        : {df.shape[0]:>8,} rows  x  {df.shape[1]} columns")
    print(f"  Duplicates   : {df.duplicated().sum():>8,}")
    print(f"\n  --- Nulls & Dtypes ---")
    info = pd.DataFrame({
        'dtype':    df.dtypes,
        'nulls':    df.isnull().sum(),
        'null_%':   (df.isnull().mean()*100).round(2),
        'unique':   df.nunique()
    })
    print(info.to_string())
    print(f"\n  --- Sample (2 rows) ---")
    print(df.head(2).to_string())
    print()

dfs = {}
for name, fname in files.items():
    dfs[name] = pd.read_csv(RAW / fname)

print("All files loaded.")

All files loaded.


Profile orders

In [2]:
profile("orders", dfs["orders"])

# Extra: value counts on status column
print("  --- order_status distribution ---")
print(dfs["orders"]["order_status"].value_counts())

# Extra: check all 5 timestamp columns
ts_cols = [c for c in dfs["orders"].columns if "timestamp" in c or "date" in c]
print(f"\n  Timestamp columns: {ts_cols}")
print(dfs["orders"][ts_cols].dtypes)
print(dfs["orders"][ts_cols].isnull().sum())


  ORDERS
  Shape        :   99,441 rows  x  8 columns
  Duplicates   :        0

  --- Nulls & Dtypes ---
                                dtype  nulls  null_%  unique
order_id                       object      0    0.00   99441
customer_id                    object      0    0.00   99441
order_status                   object      0    0.00       8
order_purchase_timestamp       object      0    0.00   98875
order_approved_at              object    160    0.16   90733
order_delivered_carrier_date   object   1783    1.79   81018
order_delivered_customer_date  object   2965    2.98   95664
order_estimated_delivery_date  object      0    0.00     459

  --- Sample (2 rows) ---
                           order_id                       customer_id order_status order_purchase_timestamp    order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 

Profile order_items

In [3]:
profile("order_items", dfs["order_items"])

print("  --- price distribution ---")
print(dfs["order_items"]["price"].describe())

print("\n  --- freight_value distribution ---")
print(dfs["order_items"]["freight_value"].describe())

print("\n  --- items per order ---")
print(dfs["order_items"].groupby("order_id")["order_item_id"].max().describe())


  ORDER_ITEMS
  Shape        :  112,650 rows  x  7 columns
  Duplicates   :        0

  --- Nulls & Dtypes ---
                       dtype  nulls  null_%  unique
order_id              object      0     0.0   98666
order_item_id          int64      0     0.0      21
product_id            object      0     0.0   32951
seller_id             object      0     0.0    3095
shipping_limit_date   object      0     0.0   93318
price                float64      0     0.0    5968
freight_value        float64      0     0.0    6999

  --- Sample (2 rows) ---
                           order_id  order_item_id                        product_id                         seller_id  shipping_limit_date  price  freight_value
0  00010242fe8c5a6d1ba2dd792cb16214              1  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202  2017-09-19 09:45:35   58.9          13.29
1  00018f77f2f0320c557190d7a144bdd3              1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36  2

Profile payments

In [4]:
profile("payments", dfs["payments"])

print("  --- payment_type distribution ---")
print(dfs["payments"]["payment_type"].value_counts())

print("\n  --- payment_installments distribution ---")
print(dfs["payments"]["payment_installments"].value_counts().head(10))

print("\n  --- payment_value distribution ---")
print(dfs["payments"]["payment_value"].describe())


  PAYMENTS
  Shape        :  103,886 rows  x  5 columns
  Duplicates   :        0

  --- Nulls & Dtypes ---
                        dtype  nulls  null_%  unique
order_id               object      0     0.0   99440
payment_sequential      int64      0     0.0      29
payment_type           object      0     0.0       5
payment_installments    int64      0     0.0      24
payment_value         float64      0     0.0   29077

  --- Sample (2 rows) ---
                           order_id  payment_sequential payment_type  payment_installments  payment_value
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card                     8          99.33
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card                     1          24.39

  --- payment_type distribution ---
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

  --- payment_installments distribution ---
pa

Profile reviews

In [5]:
profile("reviews", dfs["reviews"])

print("  --- review_score distribution ---")
print(dfs["reviews"]["review_score"].value_counts().sort_index())

# Check for free text nulls specifically
print("\n  --- free text null rates ---")
for col in ["review_comment_title", "review_comment_message"]:
    if col in dfs["reviews"].columns:
        null_pct = dfs["reviews"][col].isnull().mean() * 100
        print(f"  {col}: {null_pct:.1f}% null")


  REVIEWS
  Shape        :   99,224 rows  x  7 columns
  Duplicates   :        0

  --- Nulls & Dtypes ---
                          dtype  nulls  null_%  unique
review_id                object      0    0.00   98410
order_id                 object      0    0.00   98673
review_score              int64      0    0.00       5
review_comment_title     object  87656   88.34    4527
review_comment_message   object  58247   58.70   36159
review_creation_date     object      0    0.00     636
review_answer_timestamp  object      0    0.00   98248

  --- Sample (2 rows) ---
                          review_id                          order_id  review_score review_comment_title review_comment_message review_creation_date review_answer_timestamp
0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb             4                  NaN                    NaN  2018-01-18 00:00:00     2018-01-18 21:46:59
1  80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33          

Profile customers, sellers, products

In [6]:
for name in ["customers", "sellers", "products"]:
    profile(name, dfs[name])

# Extra: products — check category nulls and dimension nulls
print("  --- products: category null check ---")
print(dfs["products"]["product_category_name"].value_counts(dropna=False).head(5))

print("\n  --- products: dimension columns null check ---")
dim_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
print(dfs["products"][dim_cols].isnull().sum())


  CUSTOMERS
  Shape        :   99,441 rows  x  5 columns
  Duplicates   :        0

  --- Nulls & Dtypes ---
                           dtype  nulls  null_%  unique
customer_id               object      0     0.0   99441
customer_unique_id        object      0     0.0   96096
customer_zip_code_prefix   int64      0     0.0   14994
customer_city             object      0     0.0    4119
customer_state            object      0     0.0      27

  --- Sample (2 rows) ---
                        customer_id                customer_unique_id  customer_zip_code_prefix          customer_city customer_state
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0                     14409                 franca             SP
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3                      9790  sao bernardo do campo             SP


  SELLERS
  Shape        :    3,095 rows  x  4 columns
  Duplicates   :        0

  --- Nulls & Dtypes ---
                

Profile geolocation and translation

In [7]:
profile("geolocation", dfs["geolocation"])

print("  --- geolocation: duplicate zip prefixes ---")
zip_dupes = dfs["geolocation"].groupby("geolocation_zip_code_prefix").size()
print(f"  Unique zip prefixes     : {zip_dupes.shape[0]:,}")
print(f"  Total rows              : {dfs['geolocation'].shape[0]:,}")
print(f"  Avg rows per zip prefix : {zip_dupes.mean():.1f}")
print(f"  Max rows for one prefix : {zip_dupes.max()}")

print("\n  --- lat/lon range check (Brazil bounding box) ---")
geo = dfs["geolocation"]
print(f"  Lat range: {geo['geolocation_lat'].min():.4f} to {geo['geolocation_lat'].max():.4f}  (Brazil: -33.7 to 5.3)")
print(f"  Lon range: {geo['geolocation_lng'].min():.4f} to {geo['geolocation_lng'].max():.4f}  (Brazil: -73.9 to -28.8)")

profile("translation", dfs["translation"])


  GEOLOCATION
  Shape        : 1,000,163 rows  x  5 columns
  Duplicates   :  261,831

  --- Nulls & Dtypes ---
                               dtype  nulls  null_%  unique
geolocation_zip_code_prefix    int64      0     0.0   19015
geolocation_lat              float64      0     0.0  717360
geolocation_lng              float64      0     0.0  717613
geolocation_city              object      0     0.0    8011
geolocation_state             object      0     0.0      27

  --- Sample (2 rows) ---
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng geolocation_city geolocation_state
0                         1037       -23.545621       -46.639292        sao paulo                SP
1                         1046       -23.546081       -46.644820        sao paulo                SP

  --- geolocation: duplicate zip prefixes ---
  Unique zip prefixes     : 19,015
  Total rows              : 1,000,163
  Avg rows per zip prefix : 52.6
  Max rows for one prefix : 1146

  --- lat/lon

Cross-file key integrity checks

In [8]:
print("="*65)
print("  KEY INTEGRITY CHECKS")
print("="*65)

# orders -> customers
orphan_customers = ~dfs["orders"]["customer_id"].isin(dfs["customers"]["customer_id"])
print(f"\n  Orders with no matching customer    : {orphan_customers.sum():,}")

# order_items -> orders
orphan_items = ~dfs["order_items"]["order_id"].isin(dfs["orders"]["order_id"])
print(f"  Order items with no matching order  : {orphan_items.sum():,}")

# order_items -> products
orphan_products = ~dfs["order_items"]["product_id"].isin(dfs["products"]["product_id"])
print(f"  Order items with no matching product: {orphan_products.sum():,}")

# order_items -> sellers
orphan_sellers = ~dfs["order_items"]["seller_id"].isin(dfs["sellers"]["seller_id"])
print(f"  Order items with no matching seller : {orphan_sellers.sum():,}")

# payments -> orders
orphan_payments = ~dfs["payments"]["order_id"].isin(dfs["orders"]["order_id"])
print(f"  Payments with no matching order     : {orphan_payments.sum():,}")

# reviews -> orders
orphan_reviews = ~dfs["reviews"]["order_id"].isin(dfs["orders"]["order_id"])
print(f"  Reviews with no matching order      : {orphan_reviews.sum():,}")

# orders with no items
orders_no_items = ~dfs["orders"]["order_id"].isin(dfs["order_items"]["order_id"])
print(f"\n  Orders with no line items           : {orders_no_items.sum():,}")

# orders with no payment
orders_no_payment = ~dfs["orders"]["order_id"].isin(dfs["payments"]["order_id"])
print(f"  Orders with no payment record       : {orders_no_payment.sum():,}")

# products with no translation
no_translation = ~dfs["products"]["product_category_name"].dropna().isin(
    dfs["translation"]["product_category_name"]
)
print(f"\n  Product categories with no English  : {no_translation.sum():,}")

  KEY INTEGRITY CHECKS

  Orders with no matching customer    : 0
  Order items with no matching order  : 0
  Order items with no matching product: 0
  Order items with no matching seller : 0
  Payments with no matching order     : 0
  Reviews with no matching order      : 0

  Orders with no line items           : 775
  Orders with no payment record       : 1

  Product categories with no English  : 13


In [9]:
# Review ID duplicates — what's actually duplicated?
dupe_review_ids = dfs["reviews"][dfs["reviews"].duplicated(subset=["review_id"], keep=False)]
print(f"Rows sharing a review_id: {len(dupe_review_ids)}")
print(dupe_review_ids.sort_values("review_id").head(10).to_string())

Rows sharing a review_id: 1603
                              review_id                          order_id  review_score review_comment_title                                                                                                                                                                      review_comment_message review_creation_date review_answer_timestamp
46678  00130cbe1f9d422698c812ed8ded1919  dfcdfc43867d1c1381bfaf62d6b9c195             1                  NaN  O cartucho "original HP" 60XL não é reconhecido pela impressora, consequentemente não funcionou. Além de ter chegado com atraso de mais de 15 dias do previsto. Preciso que seja trocado.   2018-03-07 00:00:00     2018-03-20 18:08:23
29841  00130cbe1f9d422698c812ed8ded1919  04a28263e085d399c97ae49e0b477efa             1                  NaN  O cartucho "original HP" 60XL não é reconhecido pela impressora, consequentemente não funcionou. Além de ter chegado com atraso de mais de 15 dias do previsto. Preciso que sej

In [10]:
geo = dfs["geolocation"]
print("Out-of-bounds coordinates:")
oob = geo[
    (geo["geolocation_lat"] < -33.7) | (geo["geolocation_lat"] > 5.3) |
    (geo["geolocation_lng"] < -73.9) | (geo["geolocation_lng"] > -28.8)
]
print(f"  Count: {len(oob):,}")
print(f"  Lat min/max: {oob['geolocation_lat'].min():.4f} / {oob['geolocation_lat'].max():.4f}")
print(f"  Lon min/max: {oob['geolocation_lng'].min():.4f} / {oob['geolocation_lng'].max():.4f}")

Out-of-bounds coordinates:
  Count: 31
  Lat min/max: -36.6054 / 45.0659
  Lon min/max: -101.4668 / 121.1054


In [11]:
print(dfs["payments"]["payment_type"].value_counts())

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [12]:
ghost_orders = dfs["orders"][~dfs["orders"]["order_id"].isin(dfs["order_items"]["order_id"])]
print(f"Orders with no items: {len(ghost_orders)}")
print(ghost_orders["order_status"].value_counts())

Orders with no items: 775
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


In [13]:
# Find the shipped ghost order
shipped_ghost = dfs["orders"][
    (~dfs["orders"]["order_id"].isin(dfs["order_items"]["order_id"])) &
    (dfs["orders"]["order_status"] == "shipped")
]
print(shipped_ghost.to_string())

                               order_id                       customer_id order_status order_purchase_timestamp    order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
23254  a68ce1686d536ca72bd2dadc4b8671e5  d7bed5fac093a4136216072abaf599d5      shipped      2016-10-05 01:47:40  2016-10-07 03:11:22          2016-11-07 16:37:37                           NaN           2016-12-01 00:00:00
